In [5]:
import torch
import torch.nn.functional as F
import torch.nn as nn
# import torch.optim as optim
# import torch_optimizer as optim
import torch.nn.init as init
import adabound
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split

from tqdm import tqdm
from tqdm import trange

In [6]:
df = pd.read_csv('https://raw.githubusercontent.com/zorocrit/Control_Nuclear_Spins/master/initialize13Cspin/fixed_testdata.csv?token=GHSAT0AAAAAAB5HMDTBPRR5HT7V35OWUCUWZAFRLFA')
# df

In [7]:
# y = df[['x', 'N', 'z']]
# # y

y = df[['Xtau', 'XN', 'Ztau', 'ZN']]
y

,Xtau,XN,Ztau,ZN
0,1.355255,5.0,0.092347,5.0
1,2.601582,9.0,1.599869,13.0
2,1.528623,9.0,0.374334,5.0
3,1.595862,9.0,0.989839,13.0
4,0.517431,13.0,0.233642,9.0
...,...,...,...,...
1592,3.022292,9.0,1.295029,13.0
1593,3.975000,17.0,0.198750,9.0
1594,2.835303,9.0,2.257162,9.0
1595,2.945992,9.0,0.148853,13.0


In [8]:
X = df[['Al', 'Ap']]
# X

In [12]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.10, random_state=100)

In [13]:
Xdata = list(np.array(X_train.values.tolist()))
# print(Xdata)

In [14]:
ydata = list(np.array(y_train.values.tolist()))
# print(ydata)

In [15]:
torch.manual_seed(1)

In [16]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# for reproducibility
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

In [17]:
x = torch.FloatTensor(Xdata).to(device)
y = torch.FloatTensor(ydata).to(device)

c:\Users\Administrator\anaconda3\lib\site-packages\ipykernel_launcher.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at  C:\b\abs_bao0hdcrdh\croot\pytorch_1675190257512\work\torch\csrc\utils\tensor_new.cpp:204.)
  """Entry point for launching an IPython kernel.


In [18]:
num_data = 1597

num_epoch = 250000

In [19]:
from sys import stdout


model = nn.Sequential(

    nn.Linear(2,10),

    nn.ReLU(),

    nn.Linear(10,40),

    nn.ReLU(),

    nn.Linear(40,55),

    nn.ReLU(),

    nn.Linear(55,80),

    nn.ReLU(),

    nn.Linear(80,120),

    nn.ReLU(),

    nn.Linear(120,150),

    nn.ReLU(),

    nn.Linear(150,130),

    nn.ReLU(),

    nn.Linear(130,110),

    nn.ReLU(),

    nn.Linear(110,95),

    nn.ReLU(),

    nn.Linear(95,70),

    nn.ReLU(),

    nn.Linear(70,30),

    nn.ReLU(),

    nn.Linear(30,4)

).to(device)

#ReLU라는 Activation Function을 사용하여, 4개의 Linear Layer로 모델 구현

# Input layer에 1개씩 데이터가 들어가므로 nn.Linear(1,2)이며, 최종적으로 1개의 값이 나와야하기에 Output Layer는 nn.Linear(4,1)

 

loss_func = nn.L1Loss().to(device)

# optimizer = optim.SGD(model.parameters(),lr = 0.0002)

# optimizer = adabound.AdaBound(model.parameters(), lr=0.935 * 1e-4, final_lr=0.05758)  #Loss: 0.0537

optimizer = adabound.AdaBound(model.parameters(), lr=0.435 * 1e-3, final_lr=0.05558)

loss_array = []

minloss = 10

for i in tqdm(range(num_epoch)) : 

    optimizer.zero_grad()

    output = model(x)

    loss = loss_func(output,y)

    loss.backward()

    optimizer.step()

    if(loss < minloss):
       minloss = loss

    loss_array.append(loss)

    if(i%10000 == 0):
      # print("Case "+ str(i) + ", Loss: " + str(loss.data))
      stdout.write("Minimum loss: " + str(minloss))

    if i == num_epoch - 1:

        print(loss.data)

        param_list = list(model.parameters())

        #최종 학습된 마지막 결과물의 Parameter 저장

        print(param_list)

loss_array = torch.tensor(loss_array)

loss_array.detach().numpy()

plt.plot(loss_array)

plt.show()

#Loss(y_predicted - y_real)값이 어떻게 변하는지 그래프로 도식화

  0%|          | 0/250000 [00:00<?, ?it/s]c:\Users\Administrator\anaconda3\lib\site-packages\adabound\adabound.py:94: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha) (Triggered internally at  C:\b\abs_bao0hdcrdh\croot\pytorch_1675190257512\work\torch\csrc\utils\python_arg_parser.cpp:1174.)
  exp_avg.mul_(beta1).add_(1 - beta1, grad)
  0%|          | 1/250000 [00:00<33:03:07,  2.10it/s]

Minimum loss: tensor(5.6075, grad_fn=<L1LossBackward0>)

  0%|          | 44/250000 [00:15<24:23:47,  2.85it/s]


KeyboardInterrupt: 

In [41]:
# 임의의 입력 [73, 80, 75]를 선언
new_var =  torch.FloatTensor([[4.493041, 0.5427]]).to(device)
# 입력한 값 [73, 80, 75]에 대해서 예측값 y를 리턴받아서 pred_y에 저장
pred_y = model(new_var) 
print("훈련 후 예측값 :", pred_y) 

훈련 후 예측값 : tensor([[ 3.4563, 16.9313,  1.2362, 14.3154]], device='cuda:0',
       grad_fn=<AddmmBackward0>)
